In [6]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW

from evaluate import evaluate_bert_softmax
from utils.bert.data_utils import read_conll_2, build_tag2idx, NERDataset, build_collate_fn
from models.bert_softmax import BertSoftmaxNER

In [7]:
TRAIN_PATH = "../data/matscholar/train.txt"
VALID_PATH = "../data/matscholar/valid.txt"

MODEL_NAME = "bert-base-cased"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
train_sentence, train_tags = read_conll_2(TRAIN_PATH)
valid_sentence, valid_tags = read_conll_2(VALID_PATH)

tag2idx, idx2tag = build_tag2idx(train_tags)

In [9]:
train_dataset = NERDataset(train_sentence, train_tags)
valid_dataset = NERDataset(valid_sentence, valid_tags)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

collate_fn = build_collate_fn(
    tokenizer=tokenizer,
    label2id=tag2idx,
    max_length=MAX_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

model = BertSoftmaxNER(
    model_name=MODEL_NAME,
    num_labels=len(tag2idx),
    id2label=idx2tag,
    label2id=tag2idx
).to(DEVICE)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
best_valid_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        token_type_ids = batch.get("token_type_ids")

        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            token_type_ids=token_type_ids
        )

        loss = outputs["loss"]
        loss.backward()

        # 梯度裁剪，防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    valid_loss, valid_precision, valid_recall, valid_f1, _, _ = evaluate_bert_softmax(
        model=model,
        dataloader=valid_loader,
        id2label=idx2tag,
        device=DEVICE
    )

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Valid Loss: {valid_loss:.4f}")
    print(f"Valid Precision: {valid_precision:.4f}")
    print(f"Valid Recall: {valid_recall:.4f}")
    print(f"Valid F1: {valid_f1:.4f}")

    if valid_f1 > best_valid_f1:
        best_valid_f1 = valid_f1

        torch.save({
            'model': model.state_dict(),
            'tag2idx': tag2idx
        }, '../models/bert_softmax.pt')
        print("保存当前最优模型")


Epoch 1/5
Train Loss: 0.7588
Valid Loss: 0.2486
Valid Precision: 0.7233
Valid Recall: 0.7962
Valid F1: 0.7580
保存当前最优模型

Epoch 2/5
Train Loss: 0.1975
Valid Loss: 0.2139
Valid Precision: 0.7720
Valid Recall: 0.8327
Valid F1: 0.8012
保存当前最优模型

Epoch 3/5
Train Loss: 0.1086
Valid Loss: 0.2259
Valid Precision: 0.7945
Valid Recall: 0.8705
Valid F1: 0.8308
保存当前最优模型

Epoch 4/5
Train Loss: 0.0621
Valid Loss: 0.2343
Valid Precision: 0.8121
Valid Recall: 0.8566
Valid F1: 0.8338
保存当前最优模型

Epoch 5/5
Train Loss: 0.0400
Valid Loss: 0.2480
Valid Precision: 0.8131
Valid Recall: 0.8605
Valid F1: 0.8361
保存当前最优模型
